<a href="https://colab.research.google.com/github/divyasri-jonnadula-16/spoken-language-identification/blob/main/Wav2vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets torchaudio librosa soundfile -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import torch
import librosa
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification
from sklearn.metrics import accuracy_score
from tqdm import tqdm

In [ ]:
TRAIN_DIR = "/content/drive/MyDrive/spokenvoice/train"
TEST_DIR = "/content/drive/MyDrive/spokenvoice/test"

In [ ]:
def collate_fn(batch):
    input_values = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch])

    padded_inputs = processor.pad(
        {"input_values": input_values},
        padding=True,
        return_tensors="pt"
    )

    return padded_inputs.input_values, labels

In [ ]:
label_map = {
    "english": 0,
    "german": 1,
    "spanish": 2
}

id2label = {v:k for k,v in label_map.items()}

In [ ]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=3,
    label2id=label_map,
    id2label=id2label
)

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
quantizer.codevectors        | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 
projector.weight             | MISSING    | 
projector.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
model.freeze_feature_encoder()

In [ ]:
class SpeechDataset(Dataset):
    def __init__(self, root_dir):
        self.files = []
        self.labels = []

        for lang in os.listdir(root_dir):
            lang_path = os.path.join(root_dir, lang)
            if not os.path.isdir(lang_path):
                continue

            for file in os.listdir(lang_path):
                if file.endswith(".flac"):
                    self.files.append(os.path.join(lang_path, file))
                    self.labels.append(label_map[lang])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        audio, sr = librosa.load(self.files[idx], sr=16000, duration=5)

        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        )

        return inputs.input_values.squeeze(), torch.tensor(self.labels[idx])

In [ ]:
train_dataset = SpeechDataset(TRAIN_DIR)
test_dataset = SpeechDataset(TEST_DIR)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=2, collate_fn=collate_fn)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
EPOCHS = 2

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for inputs, labels in tqdm(train_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

100%|██████████| 3000/3000 [22:48<00:00,  2.19it/s]


Epoch 1 Loss: 0.1552


100%|██████████| 3000/3000 [10:45<00:00,  4.64it/s]

Epoch 2 Loss: 0.0105


In [ ]:
model.eval()
preds = []
true = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)

        outputs = model(inputs)
        predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()

        preds.extend(predictions)
        true.extend(labels.numpy())

acc = accuracy_score(true, preds)
print("Test Accuracy:", acc)

Test Accuracy: 0.9907407407407407


In [ ]:
model.save_pretrained("/content/lang_model")
processor.save_pretrained("/content/lang_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['/content/lang_model/processor_config.json']

In [ ]:
from google.colab import files

uploaded = files.upload()
file_path = list(uploaded.keys())[0]

Saving de_f_63f5b79c76cf5a1a4bbd1c40f54b166e.fragment7.flac to de_f_63f5b79c76cf5a1a4bbd1c40f54b166e.fragment7.flac


In [ ]:
def predict_language(file_path):
    audio, sr = librosa.load(file_path, sr=16000)

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    inputs = inputs.input_values.to(device)

    with torch.no_grad():
        logits = model(inputs).logits

    pred = torch.argmax(logits, dim=1).item()
    print("Predicted Language:", id2label[pred])

predict_language(file_path)

Predicted Language: german
